# LeetCode 1973 — Count Nodes Equal to Sum of Descendants bằng Python

Link đề: https://leetcode.com/problems/count-nodes-equal-to-sum-of-descendants/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Tree

> **Lưu ý (bài Premium 🔒):** đề trong notebook được viết lại theo hiểu biết về bài (nguồn tham khảo: bản mô tả của doocs/leetcode). Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def equalToDescendants(self, root: Optional[TreeNode]) -> int:
        ...
```

Notebook đi từ **tính tổng hậu duệ riêng cho từng nút** (vét cạn `O(n²)`) → **DFS hậu thứ tự trả về tổng cây con** (đệ quy) → **hậu thứ tự lặp bằng stack** (lời giải chính, an toàn với cây lệch `10⁵` nút) → **BFS rồi duyệt ngược thứ tự** để cộng dồn từ lá lên gốc.

## 1. Mục tiêu học

1. Phân biệt **tổng hậu duệ** (không gồm chính nút) với **tổng cây con** (gồm chính nút): `desc(x) = sub(x) − x.val = sub(left) + sub(right)`.
2. Dùng DFS **hậu thứ tự**: mỗi nút nhận tổng cây con của hai con rồi trả tổng của mình lên cha — mỗi nút thăm một lần.
3. Cài đặt hậu thứ tự **lặp** và kỹ thuật "BFS rồi đi ngược" để tránh giới hạn đệ quy của Python.
4. Xử lý đúng trường hợp **lá** (tổng hậu duệ bằng 0 ⇒ chỉ lá có giá trị 0 được đếm).
5. Tự dựng `TreeNode`, dựng/xuất cây kiểu LeetCode, sinh cây ngẫu nhiên và đối chiếu các cách.

## 2. Đề bài & ví dụ

Cho gốc `root` của một cây nhị phân, trả về **số nút** có giá trị **bằng tổng giá trị các hậu duệ** của nó.

Hậu duệ của `x` là mọi nút nằm trên đường đi từ `x` xuống một lá (không tính chính `x`). Nếu nút không có hậu duệ thì tổng được coi là `0`.

**Ví dụ 1:** `root = [10,3,4,2,1]` → `2`. Nút `10`: hậu duệ `3 + 4 + 2 + 1 = 10` ✔. Nút `3`: hậu duệ `2 + 1 = 3` ✔.

**Ví dụ 2:** `root = [2,3,null,2,null]` → `0`.

**Ví dụ 3:** `root = [0]` → `1` (không có hậu duệ, tổng 0 = giá trị 0).

**Ràng buộc:** số nút trong `[1, 10⁵]`; `0 ≤ Node.val ≤ 10⁵`.

In [1]:
import random
import sys
import time
from collections import deque
from typing import Optional, List

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
    def __repr__(self):
        return f"TreeNode({self.val})"

def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()
        for side in ("left", "right"):
            if i < len(values) and values[i] is not None:
                child = TreeNode(values[i]); setattr(node, side, child); q.append(child)
            i += 1
    return root

def to_list(root):
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None); continue
        out.append(node.val); q.append(node.left); q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

examples = [([10, 3, 4, 2, 1], 2), ([2, 3, None, 2, None], 0), ([0], 1)]
for vals, exp in examples:
    print(vals, "-> khứ hồi", to_list(build_tree(vals)), "| kỳ vọng", exp)

[10, 3, 4, 2, 1] -> khứ hồi [10, 3, 4, 2, 1] | kỳ vọng 2
[2, 3, None, 2, None] -> khứ hồi [2, 3, None, 2] | kỳ vọng 0
[0] -> khứ hồi [0] | kỳ vọng 1


## 3. Kiến thức nền

### 3.1 Tổng cây con và tổng hậu duệ

`sub(x) = x.val + sub(x.left) + sub(x.right)` (với `sub(None) = 0`). Tổng hậu duệ `desc(x) = sub(x) − x.val`. Nút được đếm ⇔ `x.val == sub(x.left) + sub(x.right)`.

In [2]:
def subtree_sum(node):
    total, stack = 0, [node] if node else []
    while stack:
        u = stack.pop(); total += u.val
        if u.left: stack.append(u.left)
        if u.right: stack.append(u.right)
    return total

r = build_tree([10, 3, 4, 2, 1])
print("sub(10) =", subtree_sum(r), "| desc(10) =", subtree_sum(r) - r.val, "| desc(3) =", subtree_sum(r.left) - r.left.val)

sub(10) = 20 | desc(10) = 10 | desc(3) = 3


### 3.2 Hậu thứ tự (postorder)

Thứ tự **trái → phải → nút**: khi xử lý một nút, hai cây con đã xử lý xong ⇒ có sẵn tổng của chúng. Đây là "kết quả chảy từ lá lên gốc".

In [3]:
def postorder_vals(root):
    out, stack = [], [(root, False)]
    while stack:
        node, done = stack.pop()
        if node is None: continue
        if done: out.append(node.val)
        else: stack += [(node, True), (node.right, False), (node.left, False)]
    return out

print(postorder_vals(build_tree([10, 3, 4, 2, 1])))

[2, 1, 3, 4, 10]


### 3.3 Giới hạn đệ quy

`sys.getrecursionlimit()` mặc định khoảng 1000. Cây lệch `10⁵` nút làm DFS đệ quy tràn ⇒ dùng stack tường minh.

In [4]:
print("giới hạn đệ quy:", sys.getrecursionlimit())

giới hạn đệ quy: 1000


### 3.4 Thứ tự BFS đảo ngược

Trong thứ tự BFS, mọi nút con xuất hiện **sau** cha. Duyệt **ngược** danh sách BFS ⇒ con luôn được xử lý trước cha — một cách khác để có "thứ tự từ lá lên".

In [5]:
def bfs_order(root):
    order, q = [], deque([root])
    while q:
        u = q.popleft(); order.append(u)
        for c in (u.left, u.right):
            if c: q.append(c)
    return order

print([u.val for u in bfs_order(build_tree([10, 3, 4, 2, 1]))], "-> đảo:", [u.val for u in reversed(bfs_order(build_tree([10, 3, 4, 2, 1])))])

[10, 3, 4, 2, 1] -> đảo: [1, 2, 4, 3, 10]


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1 theo hậu thứ tự `2, 1, 3, 4, 10`:

| nút | tổng cây con trái | tổng cây con phải | tổng hậu duệ | bằng val? | tổng cây con trả lên |
|---|---|---|---|---|---|
| 2 | 0 | 0 | 0 | không (2 ≠ 0) | 2 |
| 1 | 0 | 0 | 0 | không | 1 |
| 3 | 2 | 1 | 3 | **có** | 6 |
| 4 | 0 | 0 | 0 | không | 4 |
| 10 | 6 | 4 | 10 | **có** | 20 |

⇒ đếm được `2`. Mỗi nút chỉ cần **hai con số** từ con của nó.

In [6]:
r = build_tree([10, 3, 4, 2, 1])
for u in reversed(bfs_order(r)):
    L, R = subtree_sum(u.left), subtree_sum(u.right)
    print(f"nút {u.val}: trái {L}, phải {R}, hậu duệ {L + R} -> {'✔' if L + R == u.val else '✘'}")

nút 1: trái 0, phải 0, hậu duệ 0 -> ✘
nút 2: trái 0, phải 0, hậu duệ 0 -> ✘
nút 4: trái 0, phải 0, hậu duệ 0 -> ✘
nút 3: trái 2, phải 1, hậu duệ 3 -> ✔
nút 10: trái 6, phải 4, hậu duệ 10 -> ✔


## 5. Cách 1 — Brute Force / baseline: tính tổng hậu duệ cho từng nút

Duyệt mọi nút; với mỗi nút, duyệt lại toàn bộ cây con của nó để tính tổng hậu duệ.

In [7]:
def count_brute(root):
    cnt = 0
    for u in bfs_order(root):
        if subtree_sum(u.left) + subtree_sum(u.right) == u.val:
            cnt += 1
    return cnt

for vals, exp in examples:
    print(vals, "->", count_brute(build_tree(vals)), "(kỳ vọng", exp, ")")

[10, 3, 4, 2, 1] -> 2 (kỳ vọng 2 )
[2, 3, None, 2, None] -> 0 (kỳ vọng 0 )
[0] -> 1 (kỳ vọng 1 )


**Độ phức tạp:** `Σ |cây con| = Σ (độ sâu + 1)` = `O(n · h)` — cây lệch `10⁵` nút là `~5·10⁹`: quá chậm. Bộ nhớ `O(n)`.

**Nút thắt:** tổng của một cây con bị tính lại cho **mọi tổ tiên**. Tính **một lần** theo hậu thứ tự và "báo cáo" lên cha.

## 6. Ý tưởng thuật toán / bước đột phá

**Hậu thứ tự trả tổng cây con:** `dfs(x)` = tổng cây con gốc `x`.
- `l = dfs(x.left)`, `r = dfs(x.right)`.
- **Nếu** `l + r == x.val` **thì** tăng đáp án.
- Trả về `x.val + l + r`.

**Bất biến:** khi xử lý `x`, `l` và `r` là tổng **chính xác** của hai cây con (đã xử lý xong). Mỗi nút được xử lý một lần ⇒ `O(n)`.

Để tránh tràn đệ quy: mô phỏng hậu thứ tự bằng stack `(node, done)` và lưu tổng cây con của mỗi nút trong `dict`, hoặc xử lý theo thứ tự BFS đảo ngược.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS đệ quy hậu thứ tự

In [8]:
def count_recursive(root):
    ans = 0
    def dfs(x):
        nonlocal ans
        if x is None:
            return 0
        l, r = dfs(x.left), dfs(x.right)
        if l + r == x.val:
            ans += 1
        return x.val + l + r
    dfs(root)
    return ans

class SolutionRecursive:
    def equalToDescendants(self, root: Optional[TreeNode]) -> int:
        ans = 0
        def dfs(x):
            nonlocal ans
            if x is None:
                return 0
            s = dfs(x.left) + dfs(x.right)
            ans += s == x.val
            return s + x.val
        dfs(root)
        return ans

for vals, exp in examples:
    r = build_tree(vals)
    print(vals, "->", count_recursive(r), SolutionRecursive().equalToDescendants(r), "(kỳ vọng", exp, ")")

[10, 3, 4, 2, 1] -> 2 2 (kỳ vọng 2 )
[2, 3, None, 2, None] -> 0 0 (kỳ vọng 0 )
[0] -> 1 1 (kỳ vọng 1 )


**Độ phức tạp:** `O(n)` thời gian, `O(h)` stack đệ quy (tràn với cây lệch sâu).

### 7.2 Cách 3 — Hậu thứ tự lặp bằng stack (lời giải chính)

In [9]:
def count_iterative(root):
    sub = {None: 0}
    ans = 0
    stack = [(root, False)]
    while stack:
        x, done = stack.pop()
        if x is None:
            continue
        if not done:
            stack.append((x, True))
            stack.append((x.right, False))
            stack.append((x.left, False))
        else:
            d = sub[x.left] + sub[x.right]         # tổng hậu duệ
            if d == x.val:
                ans += 1
            sub[x] = d + x.val
    return ans

for vals, exp in examples:
    print(vals, "->", count_iterative(build_tree(vals)), "(kỳ vọng", exp, ")")

[10, 3, 4, 2, 1] -> 2 (kỳ vọng 2 )
[2, 3, None, 2, None] -> 0 (kỳ vọng 0 )
[0] -> 1 (kỳ vọng 1 )


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (stack + `dict`). Không đệ quy.

### 7.3 Cách 4 — BFS rồi xử lý theo thứ tự ngược

In [10]:
def count_bfs_reverse(root):
    order = bfs_order(root)
    sub = {None: 0}
    ans = 0
    for x in reversed(order):                       # con luôn đứng sau cha trong BFS
        d = sub[x.left] + sub[x.right]
        ans += d == x.val
        sub[x] = d + x.val
    return ans

for vals, exp in examples:
    print(vals, "->", count_bfs_reverse(build_tree(vals)), "(kỳ vọng", exp, ")")

[10, 3, 4, 2, 1] -> 2 (kỳ vọng 2 )
[2, 3, None, 2, None] -> 0 (kỳ vọng 0 )
[0] -> 1 (kỳ vọng 1 )


**Độ phức tạp:** `O(n)` thời gian và bộ nhớ. Code gọn, dễ đúng.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1 — in mỗi lần một nút được "hoàn tất" (lấy ra lần thứ hai):

In [11]:
def trace_iterative(values):
    root = build_tree(values)
    sub = {None: 0}; ans = 0
    stack = [(root, False)]
    while stack:
        x, done = stack.pop()
        if x is None:
            continue
        if not done:
            stack += [(x, True), (x.right, False), (x.left, False)]
        else:
            d = sub[x.left] + sub[x.right]
            hit = d == x.val
            ans += hit
            sub[x] = d + x.val
            print(f"  hoàn tất nút {x.val:>2}: hậu duệ = {sub[x.left]} + {sub[x.right]} = {d:>2} {'== val ✔' if hit else '!= val'} | tổng cây con = {sub[x]:>2} | ans = {ans}")
    return ans

print("Ví dụ 1:"); trace_iterative([10, 3, 4, 2, 1])
print("Ví dụ 2:"); trace_iterative([2, 3, None, 2, None])

Ví dụ 1:
  hoàn tất nút  2: hậu duệ = 0 + 0 =  0 != val | tổng cây con =  2 | ans = 0
  hoàn tất nút  1: hậu duệ = 0 + 0 =  0 != val | tổng cây con =  1 | ans = 0
  hoàn tất nút  3: hậu duệ = 2 + 1 =  3 == val ✔ | tổng cây con =  6 | ans = 1
  hoàn tất nút  4: hậu duệ = 0 + 0 =  0 != val | tổng cây con =  4 | ans = 1
  hoàn tất nút 10: hậu duệ = 6 + 4 = 10 == val ✔ | tổng cây con = 20 | ans = 2
Ví dụ 2:
  hoàn tất nút  2: hậu duệ = 0 + 0 =  0 != val | tổng cây con =  2 | ans = 0
  hoàn tất nút  3: hậu duệ = 2 + 0 =  2 != val | tổng cây con =  5 | ans = 0
  hoàn tất nút  2: hậu duệ = 5 + 0 =  5 != val | tổng cây con =  7 | ans = 0


0

## 9. Code nộp LeetCode

In [12]:
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def equalToDescendants(self, root: Optional[TreeNode]) -> int:
        sub = {None: 0}                         # sub[x] = tổng cây con gốc x
        ans = 0
        stack = [(root, False)]
        while stack:
            x, done = stack.pop()
            if x is None:
                continue
            if not done:
                stack.append((x, True))
                stack.append((x.right, False))
                stack.append((x.left, False))
            else:
                desc = sub[x.left] + sub[x.right]
                if desc == x.val:
                    ans += 1
                sub[x] = desc + x.val
        return ans

for vals, exp in examples:
    assert Solution().equalToDescendants(build_tree(vals)) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Công thức.** Tập hậu duệ của `x` là hợp rời của hai cây con (đầy đủ) của `x`. Nên tổng hậu duệ `= sub(left) + sub(right)`, với `sub(None) = 0` (nút lá có tổng hậu duệ 0 đúng như đề).

**Tính đúng của `sub`.** Theo quy nạp trên chiều cao: `sub(x) = x.val + sub(left) + sub(right)`. Hậu thứ tự (đệ quy, stack `(node, done)`, hoặc BFS đảo ngược) bảo đảm khi xử lý `x` thì `sub(left)`, `sub(right)` đã có:
- stack: `(x, True)` được đẩy **trước** hai con nên lấy ra **sau** khi hai con hoàn tất;
- BFS đảo ngược: mọi con có chỉ số BFS lớn hơn cha ⇒ xuất hiện trước cha trong danh sách đảo.

Mỗi nút được kiểm điều kiện đúng một lần ⇒ đếm chính xác.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút, `h` là chiều cao.

| Cách | Time | Space |
|---|---|---|
| 1. Tính lại cho từng nút | `O(n · h)` (xấu nhất `O(n²)`) | `O(n)` |
| 2. DFS đệ quy | `O(n)` | `O(h)` stack |
| 3. Hậu thứ tự lặp | `O(n)` | `O(n)` |
| 4. BFS đảo ngược | `O(n)` | `O(n)` |

Tổng cây con tới `10⁵ · 10⁵ = 10¹⁰` — Python không tràn số; trong C++/Java cần `long`.

In [13]:
def random_tree(n, rnd, maxv=10**5):
    nodes = [TreeNode(rnd.randint(0, maxv))]
    slots = [(nodes[0], 'left'), (nodes[0], 'right')]
    for _ in range(n - 1):
        j = rnd.randrange(len(slots)); par, side = slots[j]
        slots[j] = slots[-1]; slots.pop()
        ch = TreeNode(rnd.randint(0, maxv)); setattr(par, side, ch); nodes.append(ch)
        slots += [(ch, 'left'), (ch, 'right')]
    return nodes[0]

def skewed(n, vals):
    root = TreeNode(vals[0]); cur = root
    for v in vals[1:]:
        cur.left = TreeNode(v); cur = cur.left
    return root

rnd = random.Random(1973)
R = random_tree(10**5, rnd, maxv=3)
S = skewed(10**5, [0] * 10**5)
for name, fn in [("recursive", count_recursive), ("iterative", count_iterative), ("bfs_reverse", count_bfs_reverse), ("Solution", Solution().equalToDescendants)]:
    t0 = time.perf_counter(); a = fn(R); dt = time.perf_counter() - t0
    print(f"{name:<11} cây ngẫu nhiên 10^5: {a:>6}  {dt * 1000:6.1f} ms")
for name, fn in [("iterative", count_iterative), ("bfs_reverse", count_bfs_reverse), ("Solution", Solution().equalToDescendants)]:
    t0 = time.perf_counter(); a = fn(S); dt = time.perf_counter() - t0
    print(f"{name:<11} cây lệch 10^5 (toàn 0): {a:>6}  {dt * 1000:6.1f} ms")

recursive   cây ngẫu nhiên 10^5:  14759    12.9 ms
iterative   cây ngẫu nhiên 10^5:  14759    34.6 ms
bfs_reverse cây ngẫu nhiên 10^5:  14759    39.7 ms
Solution    cây ngẫu nhiên 10^5:  14759    33.2 ms
iterative   cây lệch 10^5 (toàn 0): 100000    94.1 ms


bfs_reverse cây lệch 10^5 (toàn 0): 100000    20.4 ms
Solution    cây lệch 10^5 (toàn 0): 100000    35.2 ms


## 12. Bộ test

In [14]:
ALL = {
    "brute": count_brute,
    "recursive": count_recursive,
    "iterative": count_iterative,
    "bfs_reverse": count_bfs_reverse,
    "SolRec": lambda r: SolutionRecursive().equalToDescendants(r),
    "Solution": lambda r: Solution().equalToDescendants(r),
}

def check(root, exp=None, skip=()):
    res = {name: fn(root) for name, fn in ALL.items() if name not in skip}
    assert len(set(res.values())) == 1, res
    v = next(iter(res.values()))
    if exp is not None:
        assert v == exp, (v, exp)
    return v

for vals, exp in examples:
    check(build_tree(vals), exp)
print("Ví dụ đề: OK")

check(build_tree([5]), 0)                         # lá khác 0
check(build_tree([0, 0, 0]), 3)                   # mọi nút 0
check(build_tree([3, 1, 2]), 1)                   # chỉ gốc
check(build_tree([6, 3, 3, 1, 2, 0, 3]), 3)       # tự dựng: hai nút 3 và lá 0
check(build_tree([1, None, 1, None, 1]), 1)       # cây lệch phải: nút giữa có hậu duệ 1
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


Edge case `[6,3,3,1,2,0,3]` là **tự dựng**: đáp án được brute force xác nhận trong `check` (đối chiếu lại đề gốc nếu muốn). Chi tiết: nút `3` (trái) có hậu duệ `1 + 2 = 3` ✔, nút `3` (phải) có hậu duệ `0 + 3 = 3` ✔, lá `0` ✔, gốc `6` có hậu duệ `12` ✘ ⇒ `3`.

In [15]:
rnd = random.Random(2004)
for _ in range(1500):
    n = rnd.randint(1, 25)
    check(random_tree(n, rnd, maxv=rnd.choice([0, 2, 5, 10**5])))
print("1500 cây ngẫu nhiên nhỏ (giá trị nhỏ để có nhiều nút thỏa): OK")

S = skewed(5000, [rnd.randint(0, 2) for _ in range(5000)])
check(S, skip=("recursive", "SolRec"))
print("Cây lệch 5000 nút (bỏ bản đệ quy): OK")

1500 cây ngẫu nhiên nhỏ (giá trị nhỏ để có nhiều nút thỏa): OK


Cây lệch 5000 nút (bỏ bản đệ quy): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — So sánh với tổng cây con (gồm chính nút)

In [16]:
def count_bug_include_self(root):
    ans = 0
    def dfs(x):
        nonlocal ans
        if x is None:
            return 0
        s = x.val + dfs(x.left) + dfs(x.right)
        ans += s == x.val                       # SAI: s gồm cả x.val
        return s
    dfs(root)
    return ans

r = build_tree([10, 3, 4, 2, 1])
print("sai :", count_bug_include_self(r), "| đúng:", Solution().equalToDescendants(r))

sai : 3 | đúng: 2


**Vì sao sai:** `s == x.val` ⇔ tổng hậu duệ bằng 0 — chỉ đếm các nút có hậu duệ toàn 0. **Sửa:** so `x.val` với `sub(left) + sub(right)`.

### Lỗi 2 — Chỉ cộng giá trị hai con trực tiếp (không phải toàn bộ hậu duệ)

In [17]:
def count_bug_children_only(root):
    cnt = 0
    for u in bfs_order(root):
        s = (u.left.val if u.left else 0) + (u.right.val if u.right else 0)   # SAI: chỉ con trực tiếp
        cnt += s == u.val
    return cnt

r = build_tree([10, 3, 4, 2, 1])
print("sai :", count_bug_children_only(r), "| đúng:", Solution().equalToDescendants(r))

sai : 1 | đúng: 2


**Vì sao sai:** gốc `10` có con `3 + 4 = 7 ≠ 10`, nhưng hậu duệ là `3 + 4 + 2 + 1 = 10`. **Sửa:** dùng tổng **cây con** của con.

### Lỗi 3 — Bỏ qua lá (coi lá không bao giờ được đếm)

In [18]:
def count_bug_skip_leaves(root):
    ans = 0
    def dfs(x):
        nonlocal ans
        if x is None:
            return 0
        l, r = dfs(x.left), dfs(x.right)
        if (x.left or x.right) and l + r == x.val:   # SAI: lá giá trị 0 phải được đếm
            ans += 1
        return x.val + l + r
    dfs(root)
    return ans

r = build_tree([0])
print("sai :", count_bug_skip_leaves(r), "| đúng:", Solution().equalToDescendants(r))

sai : 0 | đúng: 1


**Vì sao sai:** đề nói tổng hậu duệ của nút không có hậu duệ là `0` ⇒ lá có giá trị `0` thỏa (ví dụ 3). **Sửa:** không loại lá.

### Lỗi 4 — Đệ quy trên cây lệch sâu

In [19]:
S = skewed(10**5, [0] * 10**5)
try:
    count_recursive(S)
    print("Đệ quy chạy được (giới hạn đệ quy đủ lớn)")
except RecursionError:
    print("RecursionError trên cây lệch 10^5 nút (giới hạn =", sys.getrecursionlimit(), ")")
print("Bản lặp:", Solution().equalToDescendants(S))

RecursionError trên cây lệch 10^5 nút (giới hạn = 1000 )
Bản lặp: 100000


**Sửa:** hậu thứ tự lặp hoặc BFS đảo ngược.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tính một đại lượng của **cây con** cho **mọi** nút" ⇒ DFS **hậu thứ tự** trả về đại lượng đó (tổng, số nút, chiều cao…).
- "Hậu duệ" = cây con **trừ** chính nút; "tổ tiên" = đường từ gốc tới **cha**.
- Python + cây `10⁵` nút ⇒ chuẩn bị sẵn khuôn hậu thứ tự lặp `(node, done)` hoặc BFS đảo ngược.
- Lá thường là trường hợp biên cần đọc kỹ (tổng rỗng = 0).

## 15. Biến thể của bài

1. **LeetCode 508 — Most Frequent Subtree Sum:** tổng cây con mọi nút, đếm tần suất.
2. **LeetCode 563 — Binary Tree Tilt:** `|sub(left) − sub(right)|` cộng dồn.
3. **LeetCode 2265 — Count Nodes Equal to Average of Subtree:** trả về `(tổng, số nút)` cây con.
4. **Nút bằng tổng tổ tiên:** DFS **tiền thứ tự** mang theo tổng từ gốc.
5. **Cây N-ary:** cộng tổng của mọi con.

In [20]:
def count_equal_average(root):
    """LeetCode 2265: nút có giá trị bằng trung bình (làm tròn xuống) cây con của nó."""
    info = {None: (0, 0)}; ans = 0
    for x in reversed(bfs_order(root)):
        s = x.val + info[x.left][0] + info[x.right][0]
        c = 1 + info[x.left][1] + info[x.right][1]
        ans += s // c == x.val
        info[x] = (s, c)
    return ans

def count_equal_ancestors(root):
    """Biến thể 4: nút có giá trị bằng tổng các tổ tiên."""
    ans = 0; stack = [(root, 0)]
    while stack:
        x, anc = stack.pop()
        ans += x.val == anc
        for c in (x.left, x.right):
            if c: stack.append((c, anc + x.val))
    return ans

print("2265 [4,8,5,0,1,null,6] ->", count_equal_average(build_tree([4, 8, 5, 0, 1, None, 6])))
print("tổng tổ tiên [2,2,3,4,1] ->", count_equal_ancestors(build_tree([2, 2, 3, 4, 1])))

2265 [4,8,5,0,1,null,6] -> 5
tổng tổ tiên [2,2,3,4,1] -> 2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tính lại | tổng hậu duệ riêng từng nút | `O(n·h)` | `O(n)` | đối chiếu |
| 2. DFS đệ quy | hậu thứ tự trả tổng cây con | `O(n)` | `O(h)` | code ngắn, cây nông |
| 3. Hậu thứ tự lặp | stack `(node, done)` + dict | `O(n)` | `O(n)` | **nộp bài** |
| 4. BFS đảo ngược | xử lý con trước cha | `O(n)` | `O(n)` | gọn, không đệ quy |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().equalToDescendants(build_tree([4, 2, 2, 1, 1, 1, 1]))`?

**Bài 2 (tính tay):** Lập bảng hậu thứ tự (như mục 4) cho `[5, 2, 3, None, None, 1, 2]`.

**Bài 3 (điền TODO):** Hoàn thiện `equal_nodes(root)` trả về **danh sách giá trị** các nút thỏa (theo hậu thứ tự).

**Bài 4 (tự viết lại):** Viết lại lời giải cho **cây N-ary** (`NNode(val, children)`), bản lặp.

**Bài 5 (mini challenge):** Đếm số nút mà **tổng cây con trái** bằng **tổng cây con phải** (cây con rỗng có tổng 0), trong `O(n)`, kiểm bằng brute force.

In [21]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def equal_nodes(root):
    # TODO: như Solution nhưng append x.val khi desc == x.val
    pass

# Bài 4 — TODO
class NNode:
    def __init__(self, val, children=None):
        self.val = val
        self.children = children or []

def equal_to_descendants_nary(root):
    # TODO: BFS lấy thứ tự, duyệt ngược, sub[x] = x.val + sum(sub[c])
    pass

# Bài 5 — TODO
def balanced_sum_nodes(root):
    # TODO: đếm x với sub(left) == sub(right)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Các lá `1` có hậu duệ 0.
- Mức 2: Hai nút `2` có hậu duệ `1 + 1 = 2` ✔.
- Mức 3: Gốc `4` có hậu duệ `2 + 2 + 4·1 = 8` ✘ ⇒ đáp án `2`.

**Bài 2**
- Mức 1: Hậu thứ tự: `2, 1, 2, 3, 5`.
- Mức 2: Nút `3` có hậu duệ `1 + 2 = 3` ✔.
- Mức 3: Gốc `5`: hậu duệ `2 + 6 = 8` ✘.

**Bài 3**
- Mức 1: Thay `ans += 1` bằng `out.append(x.val)`.
- Mức 2: Thứ tự stack cho ra hậu thứ tự.
- Mức 3: `len(out)` phải bằng `Solution()`.

**Bài 4**
- Mức 1: Thứ tự BFS vẫn bảo đảm con sau cha.
- Mức 2: `desc = sum(sub[c] for c in x.children)`.
- Mức 3: Kiểm bằng chuyển cây nhị phân sang N-ary.

**Bài 5**
- Mức 1: Cần `sub` của cả hai con.
- Mức 2: Lá: `0 == 0` ⇒ luôn thỏa.
- Mức 3: So với brute force dùng `subtree_sum`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1
r1 = Solution().equalToDescendants(build_tree([4, 2, 2, 1, 1, 1, 1]))
print("Bài 1:", r1)
assert r1 == 2 == count_brute(build_tree([4, 2, 2, 1, 1, 1, 1]))

# Bài 2
trace_iterative([5, 2, 3, None, None, 1, 2])

# Bài 3
def equal_nodes_sol(root):
    sub = {None: 0}; out = []
    stack = [(root, False)]
    while stack:
        x, done = stack.pop()
        if x is None: continue
        if not done:
            stack += [(x, True), (x.right, False), (x.left, False)]
        else:
            d = sub[x.left] + sub[x.right]
            if d == x.val: out.append(x.val)
            sub[x] = d + x.val
    return out

print("Bài 3:", equal_nodes_sol(build_tree([10, 3, 4, 2, 1])))
assert equal_nodes_sol(build_tree([10, 3, 4, 2, 1])) == [3, 10]

Bài 1: 2
  hoàn tất nút  2: hậu duệ = 0 + 0 =  0 != val | tổng cây con =  2 | ans = 0
  hoàn tất nút  1: hậu duệ = 0 + 0 =  0 != val | tổng cây con =  1 | ans = 0
  hoàn tất nút  2: hậu duệ = 0 + 0 =  0 != val | tổng cây con =  2 | ans = 0
  hoàn tất nút  3: hậu duệ = 1 + 2 =  3 == val ✔ | tổng cây con =  6 | ans = 1
  hoàn tất nút  5: hậu duệ = 2 + 6 =  8 != val | tổng cây con = 13 | ans = 1
Bài 3: [3, 10]


In [23]:
# Bài 4
def equal_to_descendants_nary_sol(root):
    order, q = [], deque([root])
    while q:
        u = q.popleft(); order.append(u); q.extend(u.children)
    sub = {}; ans = 0
    for x in reversed(order):
        d = sum(sub[c] for c in x.children)
        ans += d == x.val
        sub[x] = d + x.val
    return ans

def to_nary(node):
    if node is None: return None
    return NNode(node.val, [c for c in (to_nary(node.left), to_nary(node.right)) if c])

rnd = random.Random(4)
for _ in range(500):
    t = random_tree(rnd.randint(1, 30), rnd, maxv=3)
    assert equal_to_descendants_nary_sol(to_nary(t)) == Solution().equalToDescendants(t)
leaf = NNode(0); tri = NNode(6, [NNode(1), NNode(2), NNode(3)])
print("Bài 4 OK | N-ary 6 -> [1,2,3]:", equal_to_descendants_nary_sol(tri))

# Bài 5
def balanced_sum_nodes_sol(root):
    sub = {None: 0}; ans = 0
    for x in reversed(bfs_order(root)):
        ans += sub[x.left] == sub[x.right]
        sub[x] = x.val + sub[x.left] + sub[x.right]
    return ans

for _ in range(500):
    t = random_tree(rnd.randint(1, 30), rnd, maxv=3)
    brute = sum(subtree_sum(u.left) == subtree_sum(u.right) for u in bfs_order(t))
    assert balanced_sum_nodes_sol(t) == brute
print("Bài 5 OK | [1,2,2] ->", balanced_sum_nodes_sol(build_tree([1, 2, 2])))

Bài 4 OK | N-ary 6 -> [1,2,3]: 1
Bài 5 OK | [1,2,2] -> 3


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đại lượng cây con cho mọi nút" (BFS đảo ngược):**

```python
order, q = [], deque([root])
while q:
    u = q.popleft(); order.append(u)
    for c in (u.left, u.right):
        if c: q.append(c)
sub = {None: 0}
for x in reversed(order):               # con trước cha
    desc = sub[x.left] + sub[x.right]
    # ... kiểm điều kiện với desc ...
    sub[x] = desc + x.val
```

**Checklist:**
- [ ] So `x.val` với tổng **hậu duệ** (không gồm `x`)?
- [ ] Cộng **toàn bộ** cây con của con, không chỉ con trực tiếp?
- [ ] Lá có tổng hậu duệ `0` (lá giá trị 0 được đếm)?
- [ ] Không đệ quy cho cây lệch `10⁵` nút?
- [ ] Đã so với brute force trên cây ngẫu nhiên?
- [ ] Chữ ký đúng: `def equalToDescendants(self, root: Optional[TreeNode]) -> int`?